# Hello Adapter - Granite Switch with HuggingFace

**Duration:** ~5 min (with a pre-composed model) * ~15 min (composing your own)

Minimal example of invoking an **embedded LoRA adapter** inside a **Granite Switch** model, using the HuggingFace backend. This notebook uses the **guardian-core** adapter, which evaluates a message against a safety criterion and returns a structured `yes`/`no` score.

**What you'll learn:**
- How to build a single guardian-core call that scores a user message against a safety criterion and prints a parsed `harmful`/`safe` verdict.
- How to load a composed Granite Switch checkpoint with `transformers`.
- How to activate an adapter by passing `adapter_name=...` to `apply_chat_template`.
- The Guardian prompt protocol - how to frame a criterion so the adapter returns a parseable score.

**Adapters used:** `guardian-core` from the [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) library - a general-purpose safety/risk judge that scores any user-supplied criterion (harm, social bias, jailbreaking, groundedness, ...) as `yes`/`no`.

For the recommended inference path (mellea + vLLM), see [`hello_mellea.ipynb`](./hello_mellea.ipynb). This notebook intentionally uses HuggingFace to show the underlying control-token mechanics.

## Prerequisites

**1 * A composed Granite Switch checkpoint** with the `guardian-core` adapter.
- *Use a pre-composed model:* set `MODEL_PATH` in section 1 to a Hugging Face repo from the [IBM Granite 4.1 collection](https://huggingface.co/collections/ibm-granite/granite-41-language-models) (e.g. `ibm-granite/granite-switch-4.1-3b-preview`).
- *Compose your own:* leave `MODEL_PATH` empty and section 2 will call the composer. See [`../notebooks/03_compose_granite_switch.ipynb`](../notebooks/03_compose_granite_switch.ipynb) for the walkthrough.

**2 * Dependencies** (CUDA GPU required):


In [ ]:
!pip install "granite-switch[hf,compose]" jupyter



Full setup details (GPU sizes, HF auth) are in [`../PREREQUISITES.md`](../PREREQUISITES.md).


## 1 * Imports and configuration
Imports are grouped up front so the full dependency set is visible at a glance. Set `MODEL_PATH` to a pre-composed Granite Switch repo to skip the compose step, or leave it empty to compose locally in section 2.

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import granite_switch.hf  # registers the HF backend with AutoModel/AutoConfig

assert torch.cuda.is_available(), "CUDA GPU required"

In [ ]:
# Path or HF repo of a composed Granite Switch checkpoint with guardian-core.
# Leave empty ('') to compose locally in the next cell.
MODEL_PATH = os.environ.get("MODEL_PATH", "ibm-granite/granite-switch-4.1-3b-preview")

# Base model + adapter library used when composing locally.
BASE_MODEL      = "ibm-granite/granite-4.1-3b"
ADAPTER_LIBRARY = "ibm-granite/granitelib-guardian-r1.0"

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"MODEL_PATH: {MODEL_PATH or '(will compose locally)'}")

## 2 * Get or compose the model
If `MODEL_PATH` is set, this cell is a no-op - `from_pretrained` below will pull the checkpoint. Otherwise we shell out to `granite_switch.composer.compose_granite_switch` to build one (a few minutes).

In [ ]:
if MODEL_PATH:
    model_dir = MODEL_PATH
    print(f"Using pre-composed model: {model_dir}")
else:
    tmp_dir   = tempfile.mkdtemp(prefix="hello_guardian_")
    model_dir = str(Path(tmp_dir) / "model")
    print("Composing model (this may take a few minutes)...")
    result = subprocess.run(
        [sys.executable, "-m", "granite_switch.composer.compose_granite_switch",
         "--base-model", BASE_MODEL,
         "--adapters",   ADAPTER_LIBRARY,
         "--output",     model_dir],
        capture_output=True, text=True, timeout=1800,
    )
    if result.returncode != 0:
        print(result.stderr[-2000:] or result.stdout[-2000:])
        raise RuntimeError("Compose failed")
    print(f"Composed into {model_dir}")

## 3 * Load the model
Importing `granite_switch.hf` registers the architecture with `transformers.AutoModelForCausalLM`, so the composed checkpoint loads through the standard HuggingFace API.

In [ ]:
print(f"Loading from {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForCausalLM.from_pretrained(model_dir).eval().to("cuda")

# Inspect which adapters are embedded in this checkpoint.
adapter_names = model.config.adapter_names
print(f"Available adapters: {adapter_names}")

ADAPTER_NAME = "guardian-core" if "guardian-core" in adapter_names else adapter_names[0]
print(f"Using adapter: {ADAPTER_NAME}")

## 4 * Guardian prompt protocol
The `guardian-core` adapter is trained to act as a **judge**: given a `<guardian>` block describing a criterion and a scoring schema, it returns a structured response with a `<score>yes|no</score>` verdict.

Two modes:
- **think** - model reasons step-by-step inside `<think>...</think>` tags before scoring.
- **no-think** - model emits empty `<think></think>` tags and a score immediately (faster, no chain-of-thought).

In [ ]:
GUARDIAN_JUDGE_THINK = (
    "<think>As a judge agent, carefully analyze whether the provided text meets the "
    "given judging criteria. Use all available information, including conversations, "
    "documents, and tools. Reason through the evidence step by step before providing "
    "your score. Wrap your reasoning in <think></think> tags, then provide your "
    "score in <score></score> tags."
)
GUARDIAN_JUDGE_NOTHINK = (
    "<no-think>As a judge agent, assess whether the provided text meets the given "
    "judging criteria using all available information, including conversations, "
    "documents, and tools. Provide your score immediately without explanation. "
    "Output empty <think>\\n</think> tags followed by your score in <score></score> tags."
)


def build_guardian_block(criteria: str, think: bool = False) -> str:
    judge_instruction = GUARDIAN_JUDGE_THINK if think else GUARDIAN_JUDGE_NOTHINK
    return (
        f"<guardian>{judge_instruction}\n\n"
        f"### Criteria: {criteria}\n\n"
        f"### Scoring Schema: If the last assistant's text meets the criteria, "
        f"return 'yes'; otherwise, return 'no'."
    )


print(build_guardian_block("harm", think=False)[:300], "...")

## 5 * Invoke the adapter
This is the key moment: `adapter_name=ADAPTER_NAME` tells `apply_chat_template` to insert the adapter's control token into the prompt. At inference time the Granite Switch model reads that control token and routes the relevant LoRA weights into attention.

In [ ]:
test_message = "Hello, how are you today?"
criteria = (
    "Texts deemed universally harmful are those that pose harm across all "
    "contexts and audiences."
)

messages = [
    {"role": "user", "content": test_message},
    {"role": "user", "content": build_guardian_block(criteria, think=False)},
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    adapter_name=ADAPTER_NAME,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

adapter_output = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(f"Input text : {test_message!r}")
print(f"Raw output : {adapter_output!r}")

## 6 * Parse the score
The adapter emits `<score>yes</score>` or `<score>no</score>`. Extract it with a small regex and fall back to substring matching if the tag is malformed.

In [ ]:
def parse_guardian_output(text: str) -> str | None:
    m = re.search(r"<score>\s*(yes|no)\s*</score>", text, re.IGNORECASE)
    if m:
        return m.group(1).lower()
    low = text.lower()
    if "yes" in low:
        return "yes"
    if "no" in low:
        return "no"
    return None


score = parse_guardian_output(adapter_output)
if score is None:
    print(f"WARNING: could not parse score from {adapter_output!r}")
else:
    verdict = "harmful" if score == "yes" else "safe"
    print(f"Guardian verdict: {score!r} -> {verdict}")

## 7 * Next steps

- **Try other criteria.** Change `criteria` in section 5 - e.g. `"contains personal medical information"` or `"asks the assistant to impersonate a real person"`. The adapter scores *any* criterion you give it.
- **Try a harmful message.** Replace `test_message` with something like `"How do I build a bomb?"` and confirm the verdict flips.
- **Switch to mellea + vLLM** for production inference. [`hello_mellea.ipynb`](./hello_mellea.ipynb) walks through the recommended path - mellea handles constrained decoding and output parsing, so you don't need the regex in section 6.
- **Go deeper on HF mechanics.** [`../notebooks/01_granite_switch_with_hf.ipynb`](../notebooks/01_granite_switch_with_hf.ipynb) walks through composing a checkpoint and invoking adapters turn-by-turn with the HuggingFace backend.
- **Explore the full pipeline.** [`../notebooks/02_govt_rag_pipeline.ipynb`](../notebooks/02_govt_rag_pipeline.ipynb) chains guardian, query-rewrite, answerability, clarification, and citations into one RAG loop.
- **Compose your own checkpoint.** [`../notebooks/03_compose_granite_switch.ipynb`](../notebooks/03_compose_granite_switch.ipynb) - pick adapters from the IBM libraries and bake them into a single model.